<a href="https://colab.research.google.com/github/debrupa03/Development-of-Interactive-Cyber-Threat-Visualization-Dashboard/blob/main/SQL_Health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install pandas numpy

import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate healthcare dataset (18 rows, 10 columns)
def generate_healthcare_data():
    # Sample data lists
    patient_ids = [f'P{str(i).zfill(3)}' for i in range(1, 19)]
    names = ['John Smith', 'Emily Johnson', 'Michael Brown', 'Sarah Davis',
             'James Wilson', 'Lisa Anderson', 'David Martinez', 'Maria Garcia',
             'Robert Taylor', 'Jennifer Thomas', 'William Moore', 'Patricia Jackson',
             'Richard White', 'Linda Harris', 'Joseph Martin', 'Susan Thompson',
             'Charles Lee', 'Karen Walker']

    ages = np.random.randint(18, 85, size=18)
    genders = np.random.choice(['Male', 'Female'], size=18)

    conditions = ['Diabetes', 'Hypertension', 'Asthma', 'Heart Disease',
                  'Arthritis', 'Depression', 'Migraine', 'Obesity']
    diagnoses = [random.choice(conditions) for _ in range(18)]

    # Generate admission dates in the last 6 months
    base_date = datetime.now() - timedelta(days=180)
    admission_dates = [(base_date + timedelta(days=random.randint(0, 180))).strftime('%Y-%m-%d')
                       for _ in range(18)]

    # Hospital stay duration
    stay_days = np.random.randint(1, 15, size=18)

    # Treatment costs
    treatment_costs = np.random.randint(500, 15000, size=18)

    # Insurance status
    insurance = np.random.choice(['Yes', 'No'], size=18, p=[0.75, 0.25])

    # Blood pressure readings (systolic/diastolic)
    blood_pressure = [f"{np.random.randint(110, 160)}/{np.random.randint(70, 100)}"
                      for _ in range(18)]

    # Create DataFrame
    df = pd.DataFrame({
        'Patient_ID': patient_ids,
        'Name': names,
        'Age': ages,
        'Gender': genders,
        'Diagnosis': diagnoses,
        'Admission_Date': admission_dates,
        'Stay_Days': stay_days,
        'Treatment_Cost': treatment_costs,
        'Insurance': insurance,
        'Blood_Pressure': blood_pressure
    })

    return df

# Generate the dataset
healthcare_df = generate_healthcare_data()

# Display the dataset
print("Healthcare Dataset (18 rows x 10 columns):")
print(healthcare_df)
print(f"\nDataset shape: {healthcare_df.shape}")

Healthcare Dataset (18 rows x 10 columns):
   Patient_ID              Name  Age  Gender      Diagnosis Admission_Date  \
0        P001        John Smith   69    Male   Hypertension     2025-10-31   
1        P002     Emily Johnson   32  Female       Diabetes     2025-12-06   
2        P003     Michael Brown   78    Male      Arthritis     2025-09-18   
3        P004       Sarah Davis   38    Male  Heart Disease     2025-07-10   
4        P005      James Wilson   41    Male  Heart Disease     2025-08-18   
5        P006     Lisa Anderson   20    Male         Asthma     2026-01-03   
6        P007    David Martinez   39    Male   Hypertension     2025-10-25   
7        P008      Maria Garcia   70  Female   Hypertension     2025-10-04   
8        P009     Robert Taylor   19  Female       Migraine     2025-09-18   
9        P010   Jennifer Thomas   47  Female       Diabetes     2025-08-17   
10       P011     William Moore   55  Female       Diabetes     2025-09-02   
11       P012  Patric

In [ ]:
# Create SQLite database in memory
conn = sqlite3.connect(':memory:')

# Load the DataFrame into SQL database
healthcare_df.to_sql('healthcare', conn, index=False, if_exists='replace')

# Function to run SQL queries
def run_query(query, description):
    print(f"\n{'='*60}")
    print(f"Query: {description}")
    print(f"{'='*60}")
    print(f"SQL: {query}\n")
    result = pd.read_sql_query(query, conn)
    print(result)
    return result

# SQL Analysis Queries

# 1. View all data
query1 = "SELECT * FROM healthcare LIMIT 5"
run_query(query1, "Display first 5 patients")

# 2. Count patients by gender
query2 = """
SELECT Gender, COUNT(*) as Patient_Count
FROM healthcare
GROUP BY Gender
"""
run_query(query2, "Count patients by gender")

# 3. Average treatment cost by diagnosis
query3 = """
SELECT Diagnosis,
       COUNT(*) as Patient_Count,
       ROUND(AVG(Treatment_Cost), 2) as Avg_Cost,
       ROUND(AVG(Stay_Days), 1) as Avg_Stay_Days
FROM healthcare
GROUP BY Diagnosis
ORDER BY Avg_Cost DESC
"""
run_query(query3, "Average treatment cost and stay by diagnosis")

# 4. Patients with insurance vs without
query4 = """
SELECT Insurance,
       COUNT(*) as Count,
       ROUND(AVG(Treatment_Cost), 2) as Avg_Treatment_Cost
FROM healthcare
GROUP BY Insurance
"""
run_query(query4, "Insurance status analysis")

# 5. Patients older than 50
query5 = """
SELECT Name, Age, Diagnosis, Treatment_Cost
FROM healthcare
WHERE Age > 50
ORDER BY Age DESC
"""
run_query(query5, "Patients older than 50")

# 6. Most expensive treatments
query6 = """
SELECT Name, Diagnosis, Treatment_Cost, Stay_Days
FROM healthcare
ORDER BY Treatment_Cost DESC
LIMIT 5
"""
run_query(query6, "Top 5 most expensive treatments")

# 7. Average age by diagnosis
query7 = """
SELECT Diagnosis,
       ROUND(AVG(Age), 1) as Avg_Age,
       MIN(Age) as Min_Age,
       MAX(Age) as Max_Age
FROM healthcare
GROUP BY Diagnosis
ORDER BY Avg_Age DESC
"""
run_query(query7, "Age statistics by diagnosis")

# 8. Patients with long hospital stays (>7 days)
query8 = """
SELECT Name, Diagnosis, Stay_Days, Admission_Date
FROM healthcare
WHERE Stay_Days > 7
ORDER BY Stay_Days DESC
"""
run_query(query8, "Patients with hospital stay > 7 days")

#Close the connection
#conn.close()
#print("\n" + "="*60)
#print("Analysis Complete!")
#print("="*60)


Query: Display first 5 patients
SQL: SELECT * FROM healthcare LIMIT 5

  Patient_ID           Name  Age  Gender      Diagnosis Admission_Date  \
0       P001     John Smith   69    Male   Hypertension     2025-10-31   
1       P002  Emily Johnson   32  Female       Diabetes     2025-12-06   
2       P003  Michael Brown   78    Male      Arthritis     2025-09-18   
3       P004    Sarah Davis   38    Male  Heart Disease     2025-07-10   
4       P005   James Wilson   41    Male  Heart Disease     2025-08-18   

   Stay_Days  Treatment_Cost Insurance Blood_Pressure  
0         12           11516       Yes         154/78  
1          7            8013       Yes         138/84  
2          4            3112       Yes         154/70  
3          9            7541       Yes         134/76  
4          3           10055        No         118/93  

Query: Count patients by gender
SQL: 
SELECT Gender, COUNT(*) as Patient_Count
FROM healthcare
GROUP BY Gender


   Gender  Patient_Count
0  Femal

,Name,Diagnosis,Stay_Days,Admission_Date
0,Susan Thompson,Heart Disease,14,2025-08-02
1,John Smith,Hypertension,12,2025-10-31
2,Joseph Martin,Diabetes,12,2025-10-14
3,Karen Walker,Heart Disease,10,2025-10-05
4,Sarah Davis,Heart Disease,9,2025-07-10
5,Jennifer Thomas,Diabetes,9,2025-08-17
6,Linda Harris,Heart Disease,9,2025-08-01


In [ ]:
# 9. Total and average costs breakdown
query9 = """
SELECT
    COUNT(*) as Total_Patients,
    SUM(Treatment_Cost) as Total_Cost,
    ROUND(AVG(Treatment_Cost), 2) as Avg_Cost,
    MIN(Treatment_Cost) as Min_Cost,
    MAX(Treatment_Cost) as Max_Cost
FROM healthcare;
"""
run_query(query9, "Overall cost statistics")

# 10. Patients grouped by age ranges
query10 = """
SELECT
    CASE
        WHEN Age < 30 THEN '18-29'
        WHEN Age < 50 THEN '30-49'
        WHEN Age < 70 THEN '50-69'
        ELSE '70+'
    END as Age_Group,
    COUNT(*) as Patient_Count,
    ROUND(AVG(Treatment_Cost), 2) as Avg_Cost
FROM healthcare
GROUP BY Age_Group
ORDER BY Age_Group;
"""
run_query(query10, "Analysis by age groups")

# 11. Gender-based diagnosis distribution
query11 = """
SELECT
    Gender,
    Diagnosis,
    COUNT(*) as Count
FROM healthcare
GROUP BY Gender, Diagnosis
ORDER BY Gender, Count DESC;
"""
run_query(query11, "Diagnosis distribution by gender")

# 12. High-cost patients without insurance
query12 = """
SELECT
    Name,
    Age,
    Diagnosis,
    Treatment_Cost,
    Insurance
FROM healthcare
WHERE Insurance = 'No' AND Treatment_Cost > 5000
ORDER BY Treatment_Cost DESC;
"""
run_query(query12, "High-cost patients without insurance")

# 13. Average stay days by gender and insurance
query13 = """
SELECT
    Gender,
    Insurance,
    COUNT(*) as Patients,
    ROUND(AVG(Stay_Days), 1) as Avg_Stay,
    ROUND(AVG(Treatment_Cost), 2) as Avg_Cost
FROM healthcare
GROUP BY Gender, Insurance
ORDER BY Gender, Insurance;
"""
run_query(query13, "Stay duration by gender and insurance status")

# 14. Most recent admissions
query14 = """
SELECT
    Name,
    Diagnosis,
    Admission_Date,
    Stay_Days,
    Treatment_Cost
FROM healthcare
ORDER BY Admission_Date DESC
LIMIT 7;
"""
run_query(query14, "7 most recent hospital admissions")

# 15. Diagnosis frequency ranking
query15 = """
SELECT
    Diagnosis,
    COUNT(*) as Frequency,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM healthcare), 1) as Percentage
FROM healthcare
GROUP BY Diagnosis
ORDER BY Frequency DESC;
"""
run_query(query15, "Diagnosis frequency with percentages")

# 16. Patients with above-average costs
query16 = """
SELECT
    Name,
    Age,
    Diagnosis,
    Treatment_Cost,
    Stay_Days
FROM healthcare
WHERE Treatment_Cost > (SELECT AVG(Treatment_Cost) FROM healthcare)
ORDER BY Treatment_Cost DESC;
"""
run_query(query16, "Patients with above-average treatment costs")

# 17. Cost per day of stay
query17 = """
SELECT
    Name,
    Diagnosis,
    Treatment_Cost,
    Stay_Days,
    ROUND(Treatment_Cost * 1.0 / Stay_Days, 2) as Cost_Per_Day
FROM healthcare
ORDER BY Cost_Per_Day DESC
LIMIT 8;
"""
run_query(query17, "Cost efficiency - Cost per day of stay")

# 18. Summary statistics by insurance status
query18 = """
SELECT
    Insurance,
    COUNT(*) as Total_Patients,
    ROUND(AVG(Age), 1) as Avg_Age,
    ROUND(AVG(Stay_Days), 1) as Avg_Stay,
    ROUND(AVG(Treatment_Cost), 2) as Avg_Cost,
    SUM(Treatment_Cost) as Total_Cost
FROM healthcare
GROUP BY Insurance;
"""
run_query(query18, "Comprehensive summary by insurance status")

# 19. Find patients with specific conditions (using LIKE)
query19 = """
SELECT
    Patient_ID,
    Name,
    Age,
    Diagnosis,
    Treatment_Cost
FROM healthcare
WHERE Diagnosis LIKE '%Diabetes%' OR Diagnosis LIKE '%Heart%'
ORDER BY Treatment_Cost DESC;
"""
run_query(query19, "Patients with Diabetes or Heart Disease")

# 20. Complex query - Comparison with overall averages
query20 = """
SELECT
    h.Name,
    h.Age,
    h.Diagnosis,
    h.Treatment_Cost,
    ROUND((SELECT AVG(Treatment_Cost) FROM healthcare), 2) as Overall_Avg_Cost,
    ROUND(h.Treatment_Cost - (SELECT AVG(Treatment_Cost) FROM healthcare), 2) as Cost_Difference
FROM healthcare h
ORDER BY Cost_Difference DESC;
"""
run_query(query20, "Each patient's cost vs overall average")

# Optional: Create a summary report
print("\n" + "="*60)
print("FINAL SUMMARY REPORT")
print("="*60)

summary_query = """
SELECT
    'Total Patients' as Metric,
    COUNT(*) as Value
FROM healthcare
UNION ALL
SELECT
    'Average Age',
    ROUND(AVG(Age), 1)
FROM healthcare
UNION ALL
SELECT
    'Total Revenue',
    SUM(Treatment_Cost)
FROM healthcare
UNION ALL
SELECT
    'Patients with Insurance',
    COUNT(*)
FROM healthcare
WHERE Insurance = 'Yes'
UNION ALL
SELECT
    'Average Hospital Stay',
    ROUND(AVG(Stay_Days), 1)
FROM healthcare;
"""
run_query(summary_query, "Healthcare System Summary")

# Closing the connection
conn.close()
print("\n Database connection closed successfully!")


Query: Overall cost statistics
SQL: 
SELECT 
    COUNT(*) as Total_Patients,
    SUM(Treatment_Cost) as Total_Cost,
    ROUND(AVG(Treatment_Cost), 2) as Avg_Cost,
    MIN(Treatment_Cost) as Min_Cost,
    MAX(Treatment_Cost) as Max_Cost
FROM healthcare;


   Total_Patients  Total_Cost  Avg_Cost  Min_Cost  Max_Cost
0              18      133564   7420.22      1275     12706

Query: Analysis by age groups
SQL: 
SELECT 
    CASE 
        WHEN Age < 30 THEN '18-29'
        WHEN Age < 50 THEN '30-49'
        WHEN Age < 70 THEN '50-69'
        ELSE '70+'
    END as Age_Group,
    COUNT(*) as Patient_Count,
    ROUND(AVG(Treatment_Cost), 2) as Avg_Cost
FROM healthcare
GROUP BY Age_Group
ORDER BY Age_Group;


  Age_Group  Patient_Count  Avg_Cost
0     18-29              3   9870.33
1     30-49              7   8792.14
2     50-69              3   4958.67
3       70+              5   5506.40

Query: Diagnosis distribution by gender
SQL: 
SELECT 
    Gender,
    Diagnosis,
    COUNT(*) as Count
